# Evaluate: base model vs. sentiment-controlled model

Replaces exploration.ipynb + testing.ipynb (they were doing the same
comparison, one of them with a tagging function that didn't match how the
training data was built — see sentiment_utils.py).

In [1]:
import torch
import numpy as np
import pandas as pd
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentiment_utils import tag_text_with_sentiment_prefixes, calculate_sentiment_preservation

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
rouge_metric = evaluate.load("rouge")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [2]:
BASE_MODEL_PATH = "./models/bart_base_cnn"
CONTROLLED_MODEL_PATH = "./models/bart_sentiment_controlled"

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_PATH).to(device).eval()

controlled_tokenizer = AutoTokenizer.from_pretrained(CONTROLLED_MODEL_PATH)
controlled_model = AutoModelForSeq2SeqLM.from_pretrained(CONTROLLED_MODEL_PATH).to(device).eval()

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [3]:
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

N_SAMPLES = 10
samples = dataset["test"].select(range(N_SAMPLES))
articles = samples["article"]
references = samples["highlights"]

## Generate summaries from both models

In [4]:
def generate_summary(model, tokenizer, text, tag_input=False, max_input_length=1024, max_gen_length=140):
    prompt = tag_text_with_sentiment_prefixes(text) if tag_input else text
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_input_length).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, num_beams=4, max_length=max_gen_length, min_length=30,
                             length_penalty=2.0, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [5]:
base_preds, controlled_preds = [], []

for article in articles:
    base_preds.append(generate_summary(base_model, base_tokenizer, article, tag_input=False))
    controlled_preds.append(generate_summary(controlled_model, controlled_tokenizer, article, tag_input=True))

print("Generated", len(base_preds), "summaries from each model.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generated 10 summaries from each model.


## ROUGE scores

In [6]:
base_rouge = rouge_metric.compute(predictions=base_preds, references=references)
controlled_rouge = rouge_metric.compute(predictions=controlled_preds, references=references)

print("Base model ROUGE:", base_rouge)
print("Controlled model ROUGE:", controlled_rouge)

Base model ROUGE: {'rouge1': np.float64(0.40686185421029275), 'rouge2': np.float64(0.19725498095032518), 'rougeL': np.float64(0.3109303657303591), 'rougeLsum': np.float64(0.34195100988524807)}
Controlled model ROUGE: {'rouge1': np.float64(0.3942844958846413), 'rouge2': np.float64(0.18364483630445552), 'rougeL': np.float64(0.3015264828583277), 'rougeLsum': np.float64(0.3604017453638772)}


## Sentiment alignment: does the summary keep the source's emotional tone?

In [7]:
base_alignment = [calculate_sentiment_preservation(a, s)["sentiment_alignment_score"]
                  for a, s in zip(articles, base_preds)]
controlled_alignment = [calculate_sentiment_preservation(a, s)["sentiment_alignment_score"]
                        for a, s in zip(articles, controlled_preds)]

print(f"Base model mean sentiment alignment: {np.mean(base_alignment):.4f}")
print(f"Controlled model mean sentiment alignment: {np.mean(controlled_alignment):.4f}")

Base model mean sentiment alignment: 0.8987
Controlled model mean sentiment alignment: 0.9459


In [8]:
comparison_df = pd.DataFrame({
    "reference": references,
    "base_summary": base_preds,
    "base_alignment": base_alignment,
    "controlled_summary": controlled_preds,
    "controlled_alignment": controlled_alignment,
})
comparison_df.head()

,reference,base_summary,base_alignment,controlled_summary,controlled_alignment
0,Membership gives the ICC jurisdiction over all...,The Palestinian Authority becomes the 123rd me...,0.986331,Palestinian Authority officially becomes 123rd...,0.906870
1,"Theia, a bully breed mix, was apparently hit b...","Theia, a one-year-old bully breed mix, was hit...",0.652149,"Stray dog apparently hit by car, buried in fie...",0.951613
2,Mohammad Javad Zarif has spent more time with ...,Mohammad Javad Zarif is the Iranian foreign mi...,0.927482,Mohammad Javad Zarif is the Iranian foreign mi...,0.975722
3,17 Americans were exposed to the Ebola virus w...,The five were exposed to Ebola in Sierra Leone...,0.980756,Five Americans exposed to Ebola in Sierra Leon...,0.997340
4,Student is no longer on Duke University campus...,Duke student admits to hanging a noose from a ...,0.839028,Student admits hanging a noose from a tree nea...,0.999132


## Qualitative check on one example

Same test article used while developing this, useful to eyeball a single
example rather than only looking at aggregate scores.

In [9]:
unseen_article = """
A major security flaw was found in global banking networks today, letting hackers compromise thousands of accounts.
Financial authorities confirmed millions of dollars were drained from vulnerable infrastructure before teams could react.
Panic spread quickly across social media as users found themselves locked out of their primary savings applications.
Officials stated they are launching an urgent task force alongside cyber security experts to locate vulnerabilities.
The central reserve announced it will provide temporary collateral support to impacted local credit networks.
Engineers believe that patch configurations can be finalized and pushed safely by tomorrow morning.
Fortunately, tech developers successfully isolated the exploit path, confirming no structural records were corrupted.
Most institutions report that consumer balances are fully insured and missing assets will be fully restored.
Stocks rebounded dramatically late in the afternoon following news of the comprehensive rescue plan.
"""

tagged_prompt = tag_text_with_sentiment_prefixes(unseen_article)
summary = generate_summary(controlled_model, controlled_tokenizer, unseen_article, tag_input=True)

print("=== TAGGED INPUT ===")
print(tagged_prompt)
print("\n=== GENERATED SUMMARY ===")
print(summary)

metrics = calculate_sentiment_preservation(unseen_article, summary)
print(f"\nSentiment alignment score: {metrics['sentiment_alignment_score']:.4f}")

=== TAGGED INPUT ===
[NEGATIVE] A major security flaw was found in global banking networks today, letting hackers compromise thousands of accounts. Financial authorities confirmed millions of dollars were drained from vulnerable infrastructure before teams could react. Panic spread quickly across social media as users found themselves locked out of their primary savings applications. [NEUTRAL] Officials stated they are launching an urgent task force alongside cyber security experts to locate vulnerabilities. The central reserve announced it will provide temporary collateral support to impacted local credit networks. Engineers believe that patch configurations can be finalized and pushed safely by tomorrow morning. [POSITIVE] Fortunately, tech developers successfully isolated the exploit path, confirming no structural records were corrupted. Most institutions report that consumer balances are fully insured and missing assets will be fully restored. Stocks rebounded dramatically late in 